# Synthetic NIAH Counting v5

One shared v2-style GPT-2 LM with learned absolute positional embeddings and a mixed thinking/non-thinking toggle.

## Colab Setup

In [1]:
import os, subprocess, sys, pathlib

REPO_URL = ""  # Optional: set to your GitHub repo URL before running in a fresh Colab.
REPO_DIR = pathlib.Path('/content/Synthetic_NiaH_like_Count')
if 'google.colab' in sys.modules and REPO_URL and not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
print('cwd =', pathlib.Path.cwd())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])

cwd = /content


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt']' returned non-zero exit status 1.

## Runtime Settings

In [ ]:
PRESET = 'debug'  # 'debug' for artifact smoke test, 'main' for full run
STAGE = 'all'
DEVICE = 'cuda'  # change to 'cpu' if needed
OUT_ROOT = 'outputs/v5'
RUN_NAME = ''
ABLATE_NO_CONFLICT_MASK = False
TRACE_INDICES = False

args = [sys.executable, '-m', 'synthetic_niah_v5.run_v5', '--preset', PRESET, '--stage', STAGE, '--device', DEVICE, '--out-root', OUT_ROOT]
if RUN_NAME:
    args += ['--run-name', RUN_NAME]
if ABLATE_NO_CONFLICT_MASK:
    args += ['--ablate-no-conflict-mask']
if TRACE_INDICES:
    args += ['--trace-indices']
print(' '.join(args))

## Run Pipeline

In [ ]:
subprocess.check_call(args)
RUN_DIR = pathlib.Path(OUT_ROOT) / RUN_NAME if RUN_NAME else pathlib.Path(OUT_ROOT)
RUN_DIR

## Key Tables

In [ ]:
import pandas as pd

train_log = pd.read_csv(RUN_DIR / 'tables' / 'train_log.csv')
eval_by_step = pd.read_csv(RUN_DIR / 'tables' / 'eval_by_step.csv')
ambiguous = pd.read_csv(RUN_DIR / 'tables' / 'ambiguous_prefix.csv')
display(train_log.tail())
display(eval_by_step.tail(12))
display(ambiguous.tail())

## Key Figures

In [ ]:
from IPython.display import Image, display

for name in [
    'train_loss_by_step_and_mode.png',
    'final_accuracy_by_step_mode.png',
    'final_accuracy_by_count_mode.png',
    'trace_metrics_by_count.png',
    'ambiguous_prefix_probs_by_step.png',
]:
    path = RUN_DIR / 'figures' / name
    if path.exists():
        display(Image(filename=str(path)))

## Save to Google Drive

In [ ]:
DRIVE_SAVE_COMPLETED = False
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = pathlib.Path('/content/drive/MyDrive/synthetic_niah_v5')
    drive_dir.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['bash', '-lc', f'cp -r {RUN_DIR} {drive_dir}/'])
    DRIVE_SAVE_COMPLETED = True
    print('Saved to', drive_dir)
else:
    print('Not in Colab; skipping Google Drive save.')

## Auto-disconnect Colab Runtime

This cell runs immediately after the Google Drive save cell. It only disconnects when a Drive save was confirmed; local VSCode/Jupyter runs are not force-closed by default.

In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = True
FORCE_LOCAL_KERNEL_SHUTDOWN = False

if AUTO_DISCONNECT_AFTER_DRIVE_SAVE and globals().get("DRIVE_SAVE_COMPLETED", False):
    import time

    print("Google Drive save completed. Flushing Drive and disconnecting Colab runtime in 3 seconds...")
    time.sleep(3)
    try:
        from google.colab import drive, runtime

        try:
            drive.flush_and_unmount()
            print("Google Drive flushed and unmounted.")
        except Exception as e:
            print(f"Drive flush/unmount skipped or failed: {e}")
        runtime.unassign()
    except Exception as e:
        print(f"Colab runtime disconnect unavailable or failed: {e}")
        if FORCE_LOCAL_KERNEL_SHUTDOWN:
            import IPython

            IPython.Application.instance().kernel.do_shutdown(restart=False)
        else:
            print("Not forcing local kernel shutdown.")
else:
    print("Auto-disconnect skipped: no confirmed Google Drive save, or AUTO_DISCONNECT_AFTER_DRIVE_SAVE is False.")

## Optional GitHub Save

In [ ]:
COMMIT_RESULTS = False
if COMMIT_RESULTS:
    subprocess.check_call(['git', 'status', '--short'])
    subprocess.check_call(['git', 'add', str(RUN_DIR)])
    subprocess.check_call(['git', 'commit', '-m', f'Add synthetic NIAH v5 {PRESET} results'])
    subprocess.check_call(['git', 'push'])